In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)


c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_core.messages import HumanMessage
response = model.invoke([HumanMessage(content="What is the capital of France?")])
print(response.content)

The capital of France is Paris.


In [9]:
from langchain_core.messages import HumanMessage,AIMessage
model.invoke([HumanMessage("Hi"),
  AIMessage("Hello!"),
  HumanMessage("What did you just say?")])

AIMessage(content='I said "Hello!" How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 54, 'total_tokens': 67, 'completion_time': 0.023834236, 'completion_tokens_details': None, 'prompt_time': 0.002616544, 'prompt_tokens_details': None, 'queue_time': 0.047096992, 'total_time': 0.02645078}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d49e3-75a7-7a23-9399-08394ee9efad-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 13, 'total_tokens': 67})

Message History

We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [11]:
##message history
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [12]:
config={"configurable": {"session_id": "user1"}}
with_message_history.invoke([HumanMessage(content="hi my name is kajal")],config=config)

AIMessage(content='Nice to meet you, Kajal. Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 42, 'total_tokens': 67, 'completion_time': 0.028016713, 'completion_tokens_details': None, 'prompt_time': 0.001963206, 'prompt_tokens_details': None, 'queue_time': 0.045446814, 'total_time': 0.029979919}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d49fe-62ea-7730-82d5-d92a15a1966b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 25, 'total_tokens': 67})

In [13]:
with_message_history.invoke([HumanMessage(content="tell me about myself?")],config=config)

AIMessage(content="However, I don't know much about you since we just started talking. Could you tell me a bit more about yourself, Kajal? \n\nYou can share your: \n- Age\n- Hobbies\n- Interests\n- Favorite things (books, movies, music, etc.)\n- Occupation or studies\n\nI'll be happy to learn more about you and maybe even give you some fun facts or suggestions based on your interests!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 81, 'total_tokens': 170, 'completion_time': 0.136384622, 'completion_tokens_details': None, 'prompt_time': 0.004647472, 'prompt_tokens_details': None, 'queue_time': 0.045391717, 'total_time': 0.141032094}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d49ff-005e-7bb2-b2a6-f30bcf37360e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'o

Prompt templates

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant answers all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="input")
    ]
)
chain=prompt | model

with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input"
)

In [ ]:
with_message_history.invoke(
    {"input": [HumanMessage(content="hi my name is abhi")]},
    config=config
)

Managing the Conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

In [ ]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

NameError: name 'model' is not defined